<table><tr>
<td style="background:#003057;color:#C29122;font-family:Georgia,serif;font-weight:700;
           font-size:15px;padding:8px 12px;border-radius:6px">GT</td>
<td style="padding-left:12px">
<b>NeuroAI: Models of the Brain and Mind</b> &nbsp;·&nbsp; PSYC 4690 / PSYC 6690<br>
<span style="color:#4a4a45">Comparing models and brains &nbsp;·&nbsp; N. Apurva Ratan Murty, PhD &nbsp;·&nbsp;
School of Psychological and Brain Sciences, Georgia Tech</span>
</td></tr></table>

# Tutorial 1 · Representational Similarity Analysis

**Do IT cortex and AlexNet arrange the same images in the same way?**

RSA sidesteps the problem that an IT site and an AlexNet unit have nothing to do with each other. Instead of
matching them up, we describe each system by the *distances between stimuli* inside it, and then ask whether
those two descriptions agree.

In this notebook you will:

1. build the **RDM** for 449 IT sites over 1379 images,
2. build an RDM for each of eight AlexNet layers,
3. **compare** them, and find which layer's geometry best matches IT,
4. **look** at both geometries directly with MDS.

Companion reading: the RSA tutorial page. Run the cells in order.

## 0 · Setup

Two things to do before anything else.

1. **Turn on a GPU.** `Runtime → Change runtime type → T4 GPU`. Everything here runs on CPU too, it is just slower.
2. **Put the data in your Google Drive.** Make a folder called `neuroai_tutorial` in *My Drive* and
   put `responses.npy` and `stimuli.zip` inside it. If you put it somewhere else, edit `DATA_DIR` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/neuroai_tutorial'   # <-- edit if your folder is elsewhere

import os
assert os.path.exists(f'{DATA_DIR}/responses.npy'), f'responses.npy not found in {DATA_DIR}'
assert os.path.exists(f'{DATA_DIR}/stimuli.zip'),   f'stimuli.zip not found in {DATA_DIR}'
print('Found the data.')

In [ ]:
# Copy the images onto the Colab machine. Reading 1379 files straight from Drive is slow.
!unzip -q -o "$DATA_DIR/stimuli.zip" -d /content/ -x "__MACOSX/*"
STIM_DIR = '/content/stimuli'
print(len(os.listdir(STIM_DIR)), 'images unzipped to', STIM_DIR)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

# ---- course plotting style -------------------------------------------------
NAVY, GOLD, RED, TEAL, GREY = '#003057', '#C29122', '#be3a2a', '#0c7a5e', '#4a4a45'
plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.edgecolor':'#c8c8c4', 'axes.linewidth':0.9,
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.titlesize':12, 'axes.titleweight':'semibold', 'axes.titlepad':10,
    'axes.labelsize':10.5, 'axes.labelcolor':'#242420',
    'xtick.color':GREY, 'ytick.color':GREY, 'xtick.labelsize':9.5, 'ytick.labelsize':9.5,
    'xtick.major.size':3.5, 'ytick.major.size':3.5,
    'legend.frameon':False, 'legend.fontsize':9.5,
    'grid.color':'#ececea', 'grid.linewidth':0.8,
    'font.size':10.5, 'figure.dpi':110, 'savefig.bbox':'tight',
})
RDM_CMAP = LinearSegmentedColormap.from_list('rdm', ['#f7f7f5', '#cfd8e0', NAVY])
print('numpy', np.__version__)

In [ ]:
def zscore(A, axis=0):
    A = np.asarray(A, float)
    mu, sd = A.mean(axis, keepdims=True), A.std(axis, keepdims=True)
    return (A - mu) / np.where(sd < 1e-12, 1.0, sd)

def pearson_cols(A, B):
    """Column-wise Pearson r between two (n, k) arrays -> (k,)."""
    A, B = A - A.mean(0), B - B.mean(0)
    den = np.sqrt((A*A).sum(0) * (B*B).sum(0))
    return np.where(den < 1e-12, 0.0, (A*B).sum(0) / den)

## 1 · The data

`responses.npy` holds recordings from **449 sites in macaque inferotemporal (IT) cortex**, measured while the
animal viewed **1379 images**. Each site has already been z-scored across images, so a value is
"how far above or below this site's average response was this image", in units of its own standard deviation.

The images are `im0001.png` … `im1379.png`, and **column *j* of `responses.npy` is image `im{j+1:04d}.png`**.
That correspondence is the whole basis of everything below, so we check it explicitly.

The stimulus set has two parts: the first 447 images are **faces** (human and monkey) and the remaining 932 are
**objects**. We will use that split throughout, both to read the plots and to build a harder test at the end.

In [ ]:
RESP = np.load(f'{DATA_DIR}/responses.npy')          # (449 sites, 1379 images)
IT   = RESP.T.astype(np.float64)                     # (1379 images, 449 sites)  <- we work image-major
N_IMG, N_SITE = IT.shape

IMG_PATHS = [f'{STIM_DIR}/im{i:04d}.png' for i in range(1, N_IMG + 1)]
assert all(os.path.exists(p) for p in IMG_PATHS), 'image / response count mismatch'

N_FACES   = 447
is_face   = np.arange(N_IMG) < N_FACES
labels    = (~is_face).astype(int)                   # 0 = face, 1 = object
CAT_NAMES = ['faces', 'objects']
CAT_COLS  = [RED, NAVY]
BOUNDS    = [0, N_FACES, N_IMG]

print(f'{N_SITE} IT sites  x  {N_IMG} images')
print(f'{is_face.sum()} faces, {(~is_face).sum()} objects')
print('each site is z-scored across images:  mean %.1e,  sd %.3f' % (IT[:,0].mean(), IT[:,0].std()))

In [ ]:
def montage(indices, ncol=8, size=92, title=''):
    rows = int(np.ceil(len(indices) / ncol))
    sheet = Image.new('RGB', (ncol*size, rows*size), 'white')
    for k, i in enumerate(indices):
        sheet.paste(Image.open(IMG_PATHS[i]).resize((size, size)), ((k % ncol)*size, (k // ncol)*size))
    fig, ax = plt.subplots(figsize=(ncol*0.85, rows*0.85 + 0.4))
    ax.imshow(sheet); ax.axis('off'); ax.set_title(title)
    return fig

rng = np.random.default_rng(0)
montage(np.r_[rng.choice(np.flatnonzero(is_face), 8, replace=False),
              rng.choice(np.flatnonzero(~is_face), 8, replace=False)],
        title='Top row: faces (images 1–447).   Bottom row: objects (images 448–1379).')
plt.show()

## 2 · The IT representational dissimilarity matrix

For every pair of images we ask: *how differently did the IT population respond to them?* We use
**1 − Pearson r** between the two 449-dimensional response patterns. Zero means the population responded
identically; larger means the two images drove IT in different ways.

The result is a 1379 × 1379 matrix. It throws away everything about *which* site did what and keeps only the
shape of the arrangement — which is exactly what makes it comparable to a model.

In [ ]:
def rdm_correlation(P):
    """P: (n_items, n_channels) -> (n_items, n_items) of 1 - Pearson r."""
    Z = zscore(P, axis=1)                 # z-score each image's pattern across channels
    C = (Z @ Z.T) / Z.shape[1]            # correlation between every pair of patterns
    return 1.0 - np.clip(C, -1, 1)

def lower_tri(D):
    i, j = np.tril_indices(D.shape[0], k=-1)
    return D[i, j]

D_IT = rdm_correlation(IT)
print('RDM:', D_IT.shape, ' range %.2f to %.2f' % (D_IT.min(), D_IT.max()))
print('unique image pairs:', len(lower_tri(D_IT)))

In [ ]:
def plot_rdm(D, title, ax=None, cmap=RDM_CMAP, vmax=None, title_color='#050504'):
    solo = ax is None
    if solo: fig, ax = plt.subplots(figsize=(5.8, 5.2))
    vmax = np.percentile(D, 98) if vmax is None else vmax
    im = ax.imshow(D, cmap=cmap, vmin=0, vmax=vmax, interpolation='nearest', aspect='equal')
    for b in BOUNDS[1:-1]:
        ax.axhline(b-.5, color='w', lw=1.8); ax.axvline(b-.5, color='w', lw=1.8)
        ax.axhline(b-.5, color=GOLD, lw=.9); ax.axvline(b-.5, color=GOLD, lw=.9)
    ctr = [(BOUNDS[i]+BOUNDS[i+1])/2 for i in range(len(CAT_NAMES))]
    ax.set_xticks(ctr); ax.set_xticklabels(CAT_NAMES)
    ax.set_yticks(ctr); ax.set_yticklabels(CAT_NAMES, rotation=90, va='center')
    ax.tick_params(length=0)
    for s in ax.spines.values(): s.set_visible(True); s.set_color('#c8c8c4')
    ax.set_title(title, color=title_color)
    if solo:
        cb = fig.colorbar(im, ax=ax, fraction=.046, pad=.03)
        cb.set_label('dissimilarity  (1 − r)', fontsize=9.5); cb.outline.set_visible(False)
        return fig
    return im

plot_rdm(D_IT, 'IT population RDM  ·  449 sites, 1379 images')
plt.show()

**Read the matrix.** The pale block in the top left is the faces: IT responds to one face much as it responds
to another. The faces-vs-objects blocks are dark, meaning the population separates those two kinds of image
strongly. The object block is mostly mid-toned with finer structure inside it.

That block structure *is* the representational geometry. The question for the rest of the notebook is whether
AlexNet has the same one.

## 2 · AlexNet

We load AlexNet with its ImageNet-trained weights and tap **eight layers**: the five convolutional stages
(after their ReLUs) and the three fully-connected ones. This is the model side of the comparison.

Two practical points:

* The raw activations are enormous — `conv1` alone gives 64 × 55 × 55 = 193,600 numbers per image. We keep a
  **fixed random sample of 2048 units per layer**. Random subsampling does not privilege any particular units,
  and both RSA and ridge regression are stable to it. Raise `MAX_UNITS` if you want to check that for yourself.
* Nothing about AlexNet is adjusted using the neural data. The network is frozen. This matters:
  it is the same logic as Yamins et al. (2014), where the model was chosen for task performance and only
  *then* compared to the brain.

In [ ]:
import torch, torchvision.transforms as T
from torchvision.models import alexnet, AlexNet_Weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = alexnet(weights=AlexNet_Weights.IMAGENET1K_V1).to(device).eval()
print('AlexNet loaded on', device)

# the eight layers we read out (ReLU outputs for conv1-5 and fc6-7, logits for fc8)
TAPS = {'conv1': net.features[1],  'conv2': net.features[4],  'conv3': net.features[7],
        'conv4': net.features[9],  'conv5': net.features[11],
        'fc6'  : net.classifier[2],'fc7'  : net.classifier[5],'fc8'  : net.classifier[6]}
LAYERS = list(TAPS)

preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),   # ImageNet statistics
])

In [ ]:
MAX_UNITS = 2048        # units kept per layer
BATCH     = 64

def extract_features(paths, max_units=MAX_UNITS, batch=BATCH, seed=0):
    buf, out, keep = {}, {k: [] for k in TAPS}, {}
    hooks = [m.register_forward_hook(
                lambda mod, inp, o, k=k: buf.__setitem__(k, o.detach().flatten(1).cpu()))
             for k, m in TAPS.items()]
    rng = np.random.default_rng(seed)
    with torch.no_grad():
        for s in range(0, len(paths), batch):
            x = torch.stack([preprocess(Image.open(p).convert('RGB'))
                             for p in paths[s:s+batch]]).to(device)
            net(x)
            for k, v in buf.items():
                v = v.numpy()
                if k not in keep:
                    n = v.shape[1]
                    keep[k] = np.sort(rng.choice(n, min(max_units, n), replace=False))
                out[k].append(v[:, keep[k]])
            if (s // batch) % 5 == 0:
                print(f'  {min(s+batch, len(paths))}/{len(paths)} images', end='\r')
    for h in hooks: h.remove()
    return {k: np.concatenate(v).astype(np.float32) for k, v in out.items()}

CACHE = f'{DATA_DIR}/alexnet_features_{MAX_UNITS}.npz'
if os.path.exists(CACHE):
    F = dict(np.load(CACHE));  print('loaded cached features from Drive')
else:
    F = extract_features(IMG_PATHS)
    np.savez_compressed(CACHE, **F);  print('\nextracted and cached to Drive')

for k in LAYERS:
    print(f'  {k:>6}: {F[k].shape}')

## 3 · One RDM per layer

Exactly the same function, applied to model features instead of neural responses. Note how the block
structure emerges as you go deeper: early layers are dominated by low-level image properties, later layers by
what the image *is*.

In [ ]:
D_LAYER = {L: rdm_correlation(F[L].astype(np.float64)) for L in LAYERS}

fig, axes = plt.subplots(3, 3, figsize=(11.5, 11.8))
plot_rdm(D_IT, 'IT  (the target)', ax=axes[0, 0], title_color=RED)
for k, L in enumerate(LAYERS):
    plot_rdm(D_LAYER[L], f'AlexNet {L}', ax=axes.flat[k+1])
fig.suptitle('Representational geometry: IT and every AlexNet layer', fontsize=13.5,
             fontweight='semibold', y=0.995)
fig.tight_layout(); plt.show()

## 4 · Comparing the geometries

Take the lower triangle of each RDM — one number per image pair — and correlate the model's vector with IT's.
We use **Spearman** rank correlation, the standard choice in RSA: we care whether the two systems *order* the
pairs the same way, not whether their dissimilarities are on the same scale.

In [ ]:
def rankdata(x):
    x = np.asarray(x, float)
    order = np.argsort(x, kind='mergesort')
    r = np.empty(len(x), float); r[order] = np.arange(1, len(x)+1)
    sx = x[order]; i = 0                                   # average ranks within ties
    while i < len(sx):
        j = i
        while j+1 < len(sx) and sx[j+1] == sx[i]: j += 1
        if j > i: r[order[i:j+1]] = (i + j + 2) / 2.0
        i = j + 1
    return r

def pearson(a, b):
    a, b = np.asarray(a,float) - np.mean(a), np.asarray(b,float) - np.mean(b)
    d = np.sqrt((a*a).sum() * (b*b).sum())
    return 0.0 if d == 0 else float((a*b).sum() / d)

def spearman(a, b):
    return pearson(rankdata(a), rankdata(b))

v_IT   = lower_tri(D_IT)
rsa_r  = np.array([spearman(v_IT, lower_tri(D_LAYER[L])) for L in LAYERS])

for L, r in zip(LAYERS, rsa_r):
    print(f'  {L:>6}   rho = {r:.3f}')
print(f'\nbest layer: {LAYERS[int(np.argmax(rsa_r))]}')

In [ ]:
def layer_bars(names, vals, title, ylab, ax=None):
    solo = ax is None
    if solo: fig, ax = plt.subplots(figsize=(7.6, 3.8))
    best = int(np.argmax(vals))
    ax.bar(np.arange(len(vals)), vals, width=.66,
           color=[GOLD if i == best else NAVY for i in range(len(vals))])
    for i, v in enumerate(vals):
        ax.text(i, v + max(vals)*.025, f'{v:.3f}', ha='center', fontsize=9,
                fontweight='600', color=GOLD if i == best else GREY)
    ax.set_xticks(np.arange(len(names))); ax.set_xticklabels(names)
    ax.set_ylabel(ylab); ax.set_title(title)
    ax.set_ylim(0, max(vals)*1.18); ax.yaxis.grid(True); ax.set_axisbelow(True)
    return fig if solo else ax

layer_bars(LAYERS, rsa_r, 'Which AlexNet layer has IT-like geometry?',
           'Spearman ρ   (model RDM vs IT RDM)')
plt.show()

**What to look for.** A rise across depth, peaking in the late convolutional or early fully-connected layers.
That ordering is the point: a model trained only to label ImageNet photographs develops, at some depth, an
arrangement of images that resembles the one in IT — and the resemblance is not uniform across the network.

Remember these are raw correlations with no ceiling (see the note at the end), so compare layers to each other
rather than to 1.0.

## 5 · Looking at the geometry directly

A number tells us the two geometries agree. It does not show us *what* the geometry is. Classical
multidimensional scaling takes an RDM and finds the 2-D arrangement of points whose distances best reproduce
it — so we can simply look at the shape both systems are being compared on.

In [ ]:
def classical_mds(D, k=2):
    n = D.shape[0]; J = np.eye(n) - np.ones((n, n))/n
    B = -0.5 * J @ (D**2) @ J
    w, V = np.linalg.eigh((B + B.T)/2)
    idx = np.argsort(w)[::-1][:k]
    return V[:, idx] * np.sqrt(np.maximum(w[idx], 0))

def plot_mds(D, title, ax):
    Y = classical_mds(D)
    for li, nm in enumerate(CAT_NAMES):
        m = labels == li
        ax.scatter(Y[m,0], Y[m,1], s=9, alpha=.62, linewidths=0, color=CAT_COLS[li], label=nm)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel('MDS 1'); ax.set_ylabel('MDS 2')
    for s in ax.spines.values(): s.set_visible(True); s.set_color('#e0e0dc')

best_layer = LAYERS[int(np.argmax(rsa_r))]
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.6))
plot_mds(D_IT,               'IT cortex',                 axes[0])
plot_mds(D_LAYER[best_layer],f'AlexNet {best_layer}  (best match)', axes[1])
plot_mds(D_LAYER['conv1'],   'AlexNet conv1  (for contrast)',       axes[2])
axes[0].legend(loc='best', markerscale=1.9)
fig.tight_layout(); plt.show()

## · A word on the noise ceiling

Every score in this notebook is a raw correlation, and raw correlations are hard to read on their own.
Measure the same IT site twice and the two measurements will not agree perfectly. No model can be expected to
predict the part of a response that does not even replicate, so the honest question is never "how close to 1.0
is this score?" but "how close to the site's own reliability is it?"

That reliability is the **noise ceiling**, and computing it needs *repeated* measurements of the same images —
which this dataset does not include. So we report raw numbers here, and you should read them as
*relative* comparisons between layers rather than as absolute statements about how good AlexNet is.

With repeat data in hand, the ceiling is short to compute:

```python
# run1, run2: (n_images, n_sites) from two independent repetitions
ceiling    = pearson_cols(run1[test], run2[test])          # per-site reliability
normalized = r / np.sqrt(np.maximum(ceiling, 1e-6))        # score read against the ceiling
```

## · Exercises

1. **Does the split drive everything?** Recompute the RSA scores using *only the 932 object images*
   (`D = rdm_correlation(IT[~is_face])`, and the same for each layer). Does the layer ordering survive when the
   faces-vs-objects distinction is removed? What does that tell you about what RSA was measuring?

2. **How much does the random unit sample matter?** Re-run `extract_features` with `seed=1` and with
   `max_units=512`. How stable are the numbers in the bar plot?

3. **An untrained control.** Load `alexnet(weights=None)` — same architecture, random weights — and repeat the
   whole comparison. How much of the IT match comes from the architecture alone, and how much from training?

4. **A different distance.** Swap `1 − r` for Euclidean distance between response patterns. Which conclusions
   change, and why might correlation distance be the more common choice for fMRI and neural data?